# **Data Exploration**

In [4]:
# Read the orders table prepared by Dataflow Gen2
orders_df = spark.read.table("dbo.staging_orders")

# Display five rows
display(orders_df.limit(5))

StatementMeta(, 81458410-f489-4afc-8df5-6556363475fb, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 54d6f7bc-b329-4c61-8bea-80995cadb889)

### Inspect the columns and data types

In [5]:
# Show all column names and their data types
orders_df.printSchema()

StatementMeta(, 81458410-f489-4afc-8df5-6556363475fb, 7, Finished, Available, Finished, False)

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)



### Count the rows

In [6]:
# Count the number of orders
print("Total rows:", orders_df.count())

# Count unique order IDs
print("Unique order IDs:", orders_df.select("order_id").distinct().count())

StatementMeta(, 81458410-f489-4afc-8df5-6556363475fb, 8, Finished, Available, Finished, False)

Total rows: 99441
Unique order IDs: 99441


### Check missing values

In [10]:
from pyspark.sql.functions import *

display(
    orders_df.select(
        [
            sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
            for c in orders_df.columns
        ]
    )
)

StatementMeta(, 81458410-f489-4afc-8df5-6556363475fb, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b8d28ed6-f22c-4190-91fc-c4b581ec4932)

### Count missing carrier dates

In [11]:
from pyspark.sql.functions import col

missing_carrier_dates = orders_df.filter(
    col("order_delivered_carrier_date").isNull()
)

print("Missing carrier dates:", missing_carrier_dates.count())

display(
    missing_carrier_dates
    .groupBy("order_status")
    .count()
    .orderBy("count", ascending=False)
)

StatementMeta(, 81458410-f489-4afc-8df5-6556363475fb, 13, Finished, Available, Finished, False)

Missing carrier dates: 1783


SynapseWidget(Synapse.DataFrame, 62447e56-c623-475f-b1b9-48d2f5b422e6)

### Check missing dates by order status

In [12]:
from pyspark.sql.functions import col

# Orders with missing approval dates
missing_approval = orders_df.filter(
    col("order_approved_at").isNull()
)

print("Missing approval dates by status:")
missing_approval.groupBy("order_status").count().show()

# Orders with missing customer delivery dates
missing_delivery = orders_df.filter(
    col("order_delivered_customer_date").isNull()
)

print("Missing delivery dates by status:")
missing_delivery.groupBy("order_status").count().show()

StatementMeta(, 81458410-f489-4afc-8df5-6556363475fb, 14, Finished, Available, Finished, False)

Missing approval dates by status:
+------------+-----+
|order_status|count|
+------------+-----+
|     Created|    5|
|    Canceled|  141|
|   Delivered|   14|
+------------+-----+

Missing delivery dates by status:
+------------+-----+
|order_status|count|
+------------+-----+
|     Shipped| 1107|
|  Processing|  301|
|    Approved|    2|
|    Invoiced|  314|
|     Created|    5|
|    Canceled|  619|
|   Delivered|    8|
| Unavailable|  609|
+------------+-----+



### Investigate the 14 delivered orders

In [14]:
# Show delivered orders with missing approval dates
display(
    orders_df.filter(
        (col("order_status") == "Delivered") &
        col("order_approved_at").isNull()
    ).select(
        "order_id",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date"
    )
)

StatementMeta(, 81458410-f489-4afc-8df5-6556363475fb, 16, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 3c8edc30-493d-4396-9fac-9b91ab9cb044)

### Check the 8 delivered orders missing customer delivery dates

In [15]:
# Find delivered orders with missing customer delivery dates
display(
    orders_df.filter(
        (col("order_status") == "Delivered") &
        col("order_delivered_customer_date").isNull()
    ).select(
        "order_id",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    )
)

StatementMeta(, 81458410-f489-4afc-8df5-6556363475fb, 17, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 1c954fbb-7ee2-44cc-8484-fc66be255c68)

### Check whether any dates are in the wrong order

In [16]:
# Find orders where the approval date is before the purchase date
display(
    orders_df.filter(
        col("order_approved_at") < col("order_purchase_timestamp")
    ).select(
        "order_id",
        "order_purchase_timestamp",
        "order_approved_at"
    )
)

StatementMeta(, 81458410-f489-4afc-8df5-6556363475fb, 18, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 68aebb5d-cc48-4c59-8dad-8dd9b63edc9b)

In [17]:
# Check the total number of orders
print("Total rows:", orders_df.count())

# Check the number of unique order IDs
print("Unique order IDs:", orders_df.select("order_id").distinct().count())

StatementMeta(, 81458410-f489-4afc-8df5-6556363475fb, 19, Finished, Available, Finished, False)

Total rows: 99441
Unique order IDs: 99441


### Create silver.orders

In [18]:
# Import the functions we need
from pyspark.sql.functions import col

# Keep the useful columns for order analysis
silver_orders = orders_df.select(
    "order_id",
    "customer_id",
    "order_status",
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
)

# Save the cleaned data as a Silver table
silver_orders.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver.orders")

StatementMeta(, 81458410-f489-4afc-8df5-6556363475fb, 20, Finished, Available, Finished, False)

### Verify the table

In [19]:
# Read the saved Silver table
silver_orders_check = spark.read.table("silver.orders")

# Confirm the row count
print("Silver orders rows:", silver_orders_check.count())

# Preview the table
display(silver_orders_check.limit(5))

StatementMeta(, 81458410-f489-4afc-8df5-6556363475fb, 21, Finished, Available, Finished, False)

Silver orders rows: 99441


SynapseWidget(Synapse.DataFrame, 0aa2ae54-bae6-4c1f-b136-12a259ba05dd)

## **Read the prepared customers table**

In [21]:
# Read the customers table prepared in Dataflow Gen2
customers_df = spark.read.table("dbo.staging_customers")

# Check the number of rows
print("Total customer rows:", customers_df.count())

# Preview the data
display(customers_df.limit(5))

StatementMeta(, 81458410-f489-4afc-8df5-6556363475fb, 23, Finished, Available, Finished, False)

Total customer rows: 99441


SynapseWidget(Synapse.DataFrame, 3b9c97c8-cd60-4f87-a239-14ca3b345102)

### Create silver.customers

In [22]:
# Keep the prepared customer columns
silver_customers = customers_df.select(
    "customer_id",
    "customer_unique_id",
    "customer_zip_code_prefix",
    "customer_city",
    "customer_state"
)

# Save the data as a Silver Delta table
silver_customers.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver.customers")

# Confirm the number of rows saved
print("Silver customers rows:", spark.read.table("silver.customers").count())

StatementMeta(, 81458410-f489-4afc-8df5-6556363475fb, 24, Finished, Available, Finished, False)

Silver customers rows: 99441


## **Read the prepared products table**

In [23]:
# Read the products table prepared in Dataflow Gen2
products_df = spark.read.table("dbo.staging_products")

# Check the number of rows
print("Total product rows:", products_df.count())

# Preview the data
display(products_df.limit(5))

StatementMeta(, 81458410-f489-4afc-8df5-6556363475fb, 25, Finished, Available, Finished, False)

Total product rows: 32951


SynapseWidget(Synapse.DataFrame, 11c0f87a-1331-4e89-a42b-8736605efe45)

## **Create silver.products**

In [24]:
# Select columns and give them clearer names
silver_products = products_df.select(
    "product_id",
    "product_category_name",
    col("product_name_lenght").alias("product_name_length"),
    col("product_description_lenght").alias("product_description_length"),
    col("product_photos_qty").alias("product_photos_quantity"),
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
)

# Save the Silver table
silver_products.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver.products")

# Verify the result
print("Silver products rows:", spark.read.table("silver.products").count())
display(spark.read.table("silver.products").limit(5))

StatementMeta(, 81458410-f489-4afc-8df5-6556363475fb, 26, Finished, Available, Finished, False)

Silver products rows: 32951


SynapseWidget(Synapse.DataFrame, 6264c0c1-59fe-48a4-9037-7aa72e2bdb09)

## **Read the prepared sellers table**

In [25]:
# Read the sellers table prepared in Dataflow Gen2
sellers_df = spark.read.table("dbo.staging_sellers")

# Check the number of rows
print("Total seller rows:", sellers_df.count())

# Preview the data
display(sellers_df.limit(5))

StatementMeta(, 81458410-f489-4afc-8df5-6556363475fb, 27, Finished, Available, Finished, False)

Total seller rows: 3095


SynapseWidget(Synapse.DataFrame, 7c8567d1-8430-440b-b687-c7e0b28e61f5)

## **Create silver.sellers**

In [26]:
# Select seller columns and use consistent names
silver_sellers = sellers_df.select(
    "seller_id",
    "seller_zip_code_prefix",
    "seller_city",
    "seller_state"
)

# Save as a Silver Delta table
silver_sellers.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver.sellers")

# Verify the saved row count
print("Silver sellers rows:", spark.read.table("silver.sellers").count())

StatementMeta(, 81458410-f489-4afc-8df5-6556363475fb, 28, Finished, Available, Finished, False)

Silver sellers rows: 3095


## **Step 1 — Read the Order_items table**

In [28]:
# Read the order_items table prepared in Dataflow Gen2
order_items_df = spark.read.table("dbo.staging_order_items")

# Check the number of rows
print("Total order item rows:", order_items_df.count())

# Preview the data
display(order_items_df.limit(5))

StatementMeta(, 81458410-f489-4afc-8df5-6556363475fb, 30, Finished, Available, Finished, False)

Total order item rows: 112650


SynapseWidget(Synapse.DataFrame, 4689715e-e604-4f26-bb2f-0507854f41d9)

### Create silver.order_items

In [29]:
# Select the prepared order item columns
silver_order_items = order_items_df.select(
    "order_id",
    "order_item_id",
    "product_id",
    "seller_id",
    "shipping_limit_date",
    "price",
    "freight_value"
)

# Save as a Silver Delta table
silver_order_items.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver.order_items")

# Verify the saved row count
print("Silver order item rows:", spark.read.table("silver.order_items").count())

StatementMeta(, 81458410-f489-4afc-8df5-6556363475fb, 31, Finished, Available, Finished, False)

Silver order item rows: 112650


## **Read the prepared table - order_payments**

In [31]:
# Read the order_payments table prepared in Dataflow Gen2
order_payments_df = spark.read.table("dbo.staging_order_payments")

# Check the number of rows
print("Total payment rows:", order_payments_df.count())

# Preview the data
display(order_payments_df.limit(5))

StatementMeta(, 81458410-f489-4afc-8df5-6556363475fb, 33, Finished, Available, Finished, False)

Total payment rows: 103886


SynapseWidget(Synapse.DataFrame, a3aeec94-2fcb-4058-8a26-76d3caae8e11)

### Create silver.order_payments

In [32]:
# Select the prepared payment columns
silver_order_payments = order_payments_df.select(
    "order_id",
    "payment_sequential",
    "payment_type",
    "payment_installments",
    "payment_value"
)

# Save as a Silver Delta table
silver_order_payments.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver.order_payments")

# Verify the saved row count
print("Silver payment rows:", spark.read.table("silver.order_payments").count())

StatementMeta(, 81458410-f489-4afc-8df5-6556363475fb, 34, Finished, Available, Finished, False)

Silver payment rows: 103886


## **Category translation**

In [42]:
# Read the prepared category translation table
category_translation_df = spark.read.table("dbo.staging_category_translation")

# Check the number of rows
print("Total category translation rows:", category_translation_df.count())

# Preview the data
display(category_translation_df.limit(10))

StatementMeta(, 81458410-f489-4afc-8df5-6556363475fb, 44, Finished, Available, Finished, False)

Total category translation rows: 71


SynapseWidget(Synapse.DataFrame, 4d9c833f-0756-4050-afdf-ae40a2c45d12)

### Create silver.category_translation


In [43]:
# Keep both category name columns
silver_category_translation = category_translation_df.select(
    "product_category_name",
    "product_category_name_english"
)

# Save as a Silver Delta table
silver_category_translation.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver.category_translation")

# Verify the saved row count
print(
    "Silver category translation rows:",
    spark.read.table("silver.category_translation").count()
)

StatementMeta(, 81458410-f489-4afc-8df5-6556363475fb, 45, Finished, Available, Finished, False)

Silver category translation rows: 71


## **Read the prepared geolocation table**

In [45]:
# Read the geolocation table prepared in Dataflow Gen2
geolocation_df = spark.read.table("dbo.staging_geolocation")

# Check the number of rows
print("Total geolocation rows:", geolocation_df.count())

# Preview the data
display(geolocation_df.limit(5))

StatementMeta(, 81458410-f489-4afc-8df5-6556363475fb, 47, Finished, Available, Finished, False)

Total geolocation rows: 1000163


SynapseWidget(Synapse.DataFrame, 1aa19508-69e7-48fd-8dde-4e78c214067c)

## **Create silver.geolocation**

In [46]:
# Select the prepared geolocation columns
silver_geolocation = geolocation_df.select(
    "geolocation_zip_code_prefix",
    "geolocation_lat",
    "geolocation_lng",
    "geolocation_city",
    "geolocation_state"
)

# Save as a Silver Delta table
silver_geolocation.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver.geolocation")

# Verify the saved row count
print(
    "Silver geolocation rows:",
    spark.read.table("silver.geolocation").count()
)

StatementMeta(, 81458410-f489-4afc-8df5-6556363475fb, 48, Finished, Available, Finished, False)

Silver geolocation rows: 1000163


## Verify all 8 Silver tables

In [1]:
# List the Silver tables we expect
silver_tables = [
    "orders",
    "customers",
    "products",
    "sellers",
    "order_items",
    "order_payments",
    "category_translation",
    "geolocation"
]

# Check each table
for table_name in silver_tables:
    table_df = spark.read.table(f"silver.{table_name}")
    print(f"{table_name}: {table_df.count()} rows")

StatementMeta(, 995e32ab-5b27-4cd4-8936-8fe9890f45af, 3, Finished, Available, Finished, False)

orders: 99441 rows
customers: 99441 rows
products: 32951 rows
sellers: 3095 rows
order_items: 112650 rows
order_payments: 103886 rows
category_translation: 71 rows
geolocation: 1000163 rows


## **Our next phase: Gold layer**

In [2]:
# Read the Silver orders table
silver_orders = spark.read.table("silver.orders")

# Preview the order data
display(silver_orders.limit(5))

# Check the available columns
print(silver_orders.columns)

StatementMeta(, 995e32ab-5b27-4cd4-8936-8fe9890f45af, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 307ad71b-65b8-4fc7-afe3-fbb9fbda6a64)

['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']


## Create the Gold fact table

In [3]:
from pyspark.sql.functions import (
    col,
    to_date,
    datediff,
    when
)

# Read the Silver orders table
silver_orders = spark.read.table("silver.orders")

# Create the Gold fact_orders table
gold_fact_orders = silver_orders.select(
    "order_id",
    "customer_id",
    "order_status",
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
).withColumn(
    "purchase_date",
    to_date(col("order_purchase_timestamp"))
).withColumn(
    "delivery_days",
    datediff(
        col("order_delivered_customer_date"),
        col("order_purchase_timestamp")
    )
).withColumn(
    "delivery_status",
    when(
        col("order_status") != "Delivered",
        "Not Delivered"
    ).when(
        col("order_delivered_customer_date").isNull(),
        "Missing Delivery Date"
    ).when(
        col("order_delivered_customer_date") >
        col("order_estimated_delivery_date"),
        "Delivered Late"
    ).otherwise(
        "Delivered On Time"
    )
)

# Save the Gold table
gold_fact_orders.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold.fact_orders")

# Verify the saved table
print("Gold fact_orders rows:", spark.read.table("gold.fact_orders").count())

display(spark.read.table("gold.fact_orders").limit(5))

StatementMeta(, 995e32ab-5b27-4cd4-8936-8fe9890f45af, 5, Finished, Available, Finished, False)

Gold fact_orders rows: 99441


SynapseWidget(Synapse.DataFrame, 218e6fe9-8839-4628-8ef1-ce2389e5be28)

In [4]:
df = spark.sql("SELECT * FROM Olist_LH.gold.fact_orders LIMIT 1000")
display(df)

StatementMeta(, 995e32ab-5b27-4cd4-8936-8fe9890f45af, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, da7ff0b2-09f8-498e-96a0-323fa3f57d97)

## Create gold.fact_sales

In [5]:
from pyspark.sql.functions import col

# Read the Silver order items table
silver_order_items = spark.read.table("silver.order_items")

# Create the Gold sales fact table
gold_fact_sales = silver_order_items.select(
    "order_id",
    "order_item_id",
    "product_id",
    "seller_id",
    "shipping_limit_date",
    "price",
    "freight_value"
).withColumn(
    "total_item_cost",
    col("price") + col("freight_value")
)

# Save the Gold table
gold_fact_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold.fact_sales")

# Verify the saved table
print("Gold fact_sales rows:", spark.read.table("gold.fact_sales").count())

display(spark.read.table("gold.fact_sales").limit(5))

StatementMeta(, 995e32ab-5b27-4cd4-8936-8fe9890f45af, 7, Finished, Available, Finished, False)

Gold fact_sales rows: 112650


SynapseWidget(Synapse.DataFrame, 2f4d4a5e-1a4a-4bc2-bbe4-995c4fe91164)

## Create gold.dim_products

In [7]:
from pyspark.sql.functions import col, lower, trim, regexp_replace

# Read the original product data
silver_products = spark.read.table("silver.products")

# Read the translation table
category_translation = spark.read.table("silver.category_translation")

# Standardize category names in both tables
products_clean = silver_products.withColumn(
    "category_key",
    lower(trim(regexp_replace(col("product_category_name"), " ", "_")))
)

translation_clean = category_translation.withColumn(
    "category_key",
    lower(trim(regexp_replace(col("product_category_name"), " ", "_")))
)


StatementMeta(, 3b79ab76-0d3b-4a23-a2f8-56f046e2897f, 9, Finished, Available, Finished, False)

In [10]:
# Read product data from Bronze to preserve original category names
silver_products = spark.read.table("silver.products")

# Read the English category translations from Silver
category_translation = spark.read.table("silver.category_translation")

# Join products to their English category names
gold_dim_products = silver_products.join(
    category_translation,
    on="product_category_name",
    how="left"
).select(
    col("product_id"),
    col("product_category_name").alias("product_category_name_portuguese"),
    col("product_category_name_english"),
    col("product_name_length"),
    col("product_description_length"),
    col("product_photos_quantity"),
    col("product_weight_g"),
    col("product_length_cm"),
    col("product_height_cm"),
    col("product_width_cm")
)

# Save the Gold product dimension
gold_dim_products.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold.dim_products")

# Verify the result
print("Gold product rows:", spark.read.table("gold.dim_products").count())

display(spark.read.table("gold.dim_products").limit(5))

StatementMeta(, 3b79ab76-0d3b-4a23-a2f8-56f046e2897f, 12, Finished, Available, Finished, False)

Gold product rows: 32951


SynapseWidget(Synapse.DataFrame, d4bbe776-267d-4851-8bb6-4f7d56dc1518)

## Create gold.dim_customers

In [9]:
# Read the Silver customers table
silver_customers = spark.read.table("silver.customers")

# Select the customer attributes for the Gold dimension
gold_dim_customers = silver_customers.select(
    "customer_id",
    "customer_unique_id",
    "customer_zip_code_prefix",
    "customer_city",
    "customer_state"
)

# Save the Gold customer dimension
gold_dim_customers.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold.dim_customers")

# Verify the saved table
print("Gold customer rows:", spark.read.table("gold.dim_customers").count())

display(spark.read.table("gold.dim_customers").limit(5))

StatementMeta(, 995e32ab-5b27-4cd4-8936-8fe9890f45af, 11, Finished, Available, Finished, False)

Gold customer rows: 99441


SynapseWidget(Synapse.DataFrame, 4717a2a3-d1e9-433e-934d-3bdf97062b7a)

## Create gold.dim_sellers

In [1]:
# Read the Silver sellers table
silver_sellers = spark.read.table("silver.sellers")

# Select the seller attributes for the Gold dimension
gold_dim_sellers = silver_sellers.select(
    "seller_id",
    "seller_zip_code_prefix",
    "seller_city",
    "seller_state"
)

# Save the Gold seller dimension
gold_dim_sellers.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold.dim_sellers")

# Verify the saved table
print("Gold seller rows:", spark.read.table("gold.dim_sellers").count())

display(spark.read.table("gold.dim_sellers").limit(5))

StatementMeta(, cc33a8e5-57ff-4062-942b-db28adf2a58e, 3, Finished, Available, Finished, False)

Gold seller rows: 3095


SynapseWidget(Synapse.DataFrame, fbc0bade-894b-411d-8fd1-ddc17fece7f0)

## Create gold.dim_date

In [3]:
from pyspark.sql.functions import *

# Read the purchase timestamps from Silver
silver_orders = spark.read.table("silver.orders")

# Find the earliest and latest purchase dates
date_range = silver_orders.select(
    min(to_date(col("order_purchase_timestamp"))).alias("min_date"),
    max(to_date(col("order_purchase_timestamp"))).alias("max_date")
).first()

# Generate every date in that range
gold_dim_date = spark.sql(
    f"""
    SELECT explode(
        sequence(
            to_date('{date_range["min_date"]}'),
            to_date('{date_range["max_date"]}'),
            interval 1 day
        )
    ) AS date
    """
)

# Add useful calendar columns
gold_dim_date = gold_dim_date \
    .withColumn("year", year(col("date"))) \
    .withColumn("quarter", quarter(col("date"))) \
    .withColumn("month_number", month(col("date"))) \
    .withColumn("month_name", date_format(col("date"), "MMMM")) \
    .withColumn("day_of_month", dayofmonth(col("date"))) \
    .withColumn("day_of_week_number", dayofweek(col("date"))) \
    .withColumn("day_name", date_format(col("date"), "EEEE"))

# Save the Gold date dimension
gold_dim_date.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold.dim_date")

# Verify the saved table
print("Gold date rows:", spark.read.table("gold.dim_date").count())

display(spark.read.table("gold.dim_date").limit(10))

StatementMeta(, cc33a8e5-57ff-4062-942b-db28adf2a58e, 5, Finished, Available, Finished, False)

Gold date rows: 774


SynapseWidget(Synapse.DataFrame, 73fa5c83-fb7b-4883-8ff0-aa99c0719cd2)

## Create gold.dim_date

In [4]:
# List the Gold tables created so far
gold_tables = [
    "fact_orders",
    "fact_sales",
    "dim_products",
    "dim_customers",
    "dim_sellers",
    "dim_date"
]

# Check each table
for table_name in gold_tables:
    table_df = spark.read.table(f"gold.{table_name}")
    print(f"{table_name}: {table_df.count()} rows")

StatementMeta(, cc33a8e5-57ff-4062-942b-db28adf2a58e, 6, Finished, Available, Finished, False)

fact_orders: 99441 rows
fact_sales: 112650 rows
dim_products: 32951 rows
dim_customers: 99441 rows
dim_sellers: 3095 rows
dim_date: 774 rows
